# Lesson 06: RAG and AI Agents — Making AI Tell the Truth

## Learning Objectives
- Understand the core principles of RAG (Retrieval-Augmented Generation)
- Build a simple RAG Q&A system from scratch with code
- Understand the concept of AI Agents: tool use, planning, memory
- Compare pure model responses with RAG-enhanced responses

> RAG is one of the most practical techniques for reducing AI hallucinations and grounding answers in evidence.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Build the Simplest RAG Q&A System

### Activity Goal
Build a micro RAG system with code: first prepare some "knowledge" pieces, then have AI retrieve relevant knowledge before answering.

The core idea of RAG is simple: **before answering, look up the relevant information first.**

The RAG trilogy:
1. **Retrieve**: find relevant content from the knowledge base
2. **Augment**: add the found content to the prompt
3. **Generate**: AI generates an answer based on the retrieved material

In [ ]:
# Activity 1: Micro RAG System

# Step 1: Prepare a knowledge base (a few "documents")
knowledge_base = [
    {'title': 'Company Overview', 'content': 'ABC Tech was founded in 2020, headquartered in San Francisco. '
     'The company focuses on AI education products, with main products including AI Learning Assistant and Smart Quiz System. '
     '2024 revenue reached $5 million, with about 200 employees.'},
    {'title': 'Product Info', 'content': 'AI Learning Assistant is ABC Tech\'s flagship product, '
     'supporting intelligent tutoring across 20+ subjects including math, English, and programming. '
     'The product uses GPT-4 as its foundation model and has over 1 million monthly active users. '
     'Pricing: Individual plan $9.99/month, Enterprise plan $199/year.'},
    {'title': 'Contact Info', 'content': 'Customer service: 1-800-AI-HELP. '
     'Business inquiries: bd@abc-tech.com. '
     'Address: 88 Innovation Drive, Suite 15, San Francisco, CA. '
     'Hours: Monday-Friday 9:00 AM - 6:00 PM.'},
    {'title': 'Funding History', 'content': 'ABC Tech completed angel round of $500K in 2021, '
     'Series A of $3M in 2022, and Series B of $15M in 2024. '
     'Series B was led by Sequoia Capital, with a valuation of $150M.'}
]

# Step 2: Simple keyword search (Retrieve)
import re

# Filler words match almost every document, so they drown out the real signal.
STOP_WORDS = {'a', 'an', 'the', 'is', 'are', 'of', 'and', 'to', 'in', 'on', 'for',
              'what', 'who', 'how', 'do', 'does', 'i', 'you', 'they', 'their', 's'}


def tokenize(text):
    """Split a sentence into matchable lowercase words, dropping filler words"""
    return [t for t in re.findall(r'[a-z0-9]+', text.lower()) if t not in STOP_WORDS]


def simple_search(query, kb, top_k=2):
    """Simplest keyword-matching retrieval: count query words found in each doc"""
    results = []
    for doc in kb:
        content = doc['content'].lower()
        score = sum(1 for word in tokenize(query) if word in content)
        if score > 0:
            results.append((score, doc))
    # Sort by relevance, then keep only the top_k documents.
    # Without the cut-off the whole knowledge base gets "retrieved", which
    # defeats the point of retrieval.
    results.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in results[:top_k]]

print(f'Knowledge base ready! {len(knowledge_base)} documents loaded.')

### Comparison Experiment: Without RAG vs With RAG

In [ ]:
# Comparison experiment: Without RAG vs With RAG

question = 'What is ABC Tech\'s customer service number? Who led their Series B funding?'

# Method 1: Without RAG (pure model response)
print('=== Method 1: Without RAG (Pure Model Response) ===')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':question}],
    temperature=0)
print(r1.choices[0].message.content)

# Method 2: With RAG
print('\n=== Method 2: With RAG (Retrieve Then Answer) ===')
# Retrieve relevant documents
relevant_docs = simple_search(question, knowledge_base)
print(f'Retrieved {len(relevant_docs)} relevant documents:')
for doc in relevant_docs:
    print(f'  - {doc["title"]}')

# Concatenate retrieved documents into context
context = '\n\n'.join([f'[{doc["title"]}]\n{doc["content"]}' for doc in relevant_docs])

# Prompt with context
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'Only answer based on the provided materials. If the information is not in the materials, say you don\'t know.'},
        {'role':'user','content':f'Reference materials:\n{context}\n\nQuestion: {question}'}
    ],
    temperature=0)
print(r2.choices[0].message.content)

print('\nCompare the two methods: Is the RAG answer more accurate and evidence-based?')

### Discussion
- How does the RAG answer differ from the pure model answer?
- When the knowledge base doesn't contain relevant information, what does RAG-mode AI say?
- What real-life scenarios are suitable for RAG? (Hint: enterprise knowledge bases, customer service FAQs, product documentation...)

---

## Activity 2: Semantic Search with Embeddings

### Activity Goal
The keyword search above is simple but not "smart" enough — "contact" and "customer service number" are semantically related but use different keywords.
We'll use OpenAI's Embedding API to turn text into "semantic vectors" for smarter retrieval.

In [ ]:
import numpy as np
# Activity 2: Semantic Search

# Not every provider offers embeddings — DeepSeek and OpenRouter currently do not.
# When they don't, fall back to a local bag-of-words vector: the code still runs,
# but semantic quality drops sharply. That is itself the lesson: an embedding's
# semantics come from model training, not from the cosine formula.
import zlib

USE_REAL_EMBEDDING = EMBEDDING_MODEL is not None
if not USE_REAL_EMBEDDING:
    print(f'Note: {PROVIDER} has no embeddings endpoint; using a local bag-of-words vector instead.')
    print('      For real semantic search switch PROVIDER to openai (or ollama + nomic-embed-text).')


def local_embedding(text, dim=512):
    """Fallback vector: hash each token into a fixed number of buckets and count"""
    vec = [0.0] * dim
    for word in tokenize(text):  # tokenize comes from Activity 1
        vec[zlib.crc32(word.encode()) % dim] += 1.0
    return vec


def get_embedding(text):
    """Vector for a piece of text: provider embeddings when available, else local"""
    if not USE_REAL_EMBEDDING:
        return local_embedding(text)
    r = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return r.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors"""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compute embeddings for each document in the knowledge base
print('Computing semantic vectors for documents...')
for doc in knowledge_base:
    doc['embedding'] = get_embedding(doc['content'])
print('Vector computation complete!')

# Semantic search
query = 'How do I get in touch with your company?'
query_emb = get_embedding(query)

# Compute similarity and sort
scored = []
for doc in knowledge_base:
    sim = cosine_similarity(query_emb, doc['embedding'])
    scored.append((sim, doc))
scored.sort(key=lambda x: x[0], reverse=True)

print(f'Query: "{query}"')
print('\nSemantic search results:')
for sim, doc in scored[:3]:
    print(f'  Similarity {sim:.3f} - {doc["title"]}')

# Answer with the most relevant document
best_doc = scored[0][1]
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'Only answer based on the provided materials.'},
        {'role':'user','content':f'Materials: {best_doc["content"]}\n\nQuestion: {query}'}
    ],
    temperature=0)
print(f'\nAI Answer: {r.choices[0].message.content}')

### Discussion
- What's the difference between semantic search and keyword search? When would you use each?
- Did "How do I get in touch with your company?" successfully match the "Contact Info" document?
- Embeddings are a core technology in RAG — do you now understand their role?

---

## Activity 3: Experience an "Agent" — Multi-Step Task

### Activity Goal
Agents don't just answer questions — they decompose tasks and execute step by step.
We'll simulate a simple agent behavior in code: think first, then act, then summarize.

In [ ]:
# Activity 3: Simple Agent Simulation

# Simulate a "Research Assistant" agent
task = 'Research the differences between GPT-4o and GPT-4o-mini, then give me a recommendation.'

# Step 1: Agent analyzes the task
print('=== Step 1: Agent Analyzes the Task ===')
step1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are a research assistant agent. Analyze the user\'s task and list the sub-steps needed to complete it.'},
        {'role':'user','content':f'Task: {task}\nList the sub-steps needed to complete this task.'}
    ],
    temperature=0.3)
plan = step1.choices[0].message.content
print(plan)

# Step 2: Agent performs research
print('\n=== Step 2: Agent Performs Research ===')
step2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are a research assistant. Based on your knowledge, provide a detailed comparison of GPT-4o and GPT-4o-mini, including capability, pricing, speed, and use cases.'},
        {'role':'user','content':'Compare GPT-4o and GPT-4o-mini'}
    ],
    temperature=0.3)
research = step2.choices[0].message.content
print(research[:400] + '...')

# Step 3: Agent gives a recommendation
print('\n=== Step 3: Agent Gives Recommendation ===')
step3 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are a research assistant. Based on the research results, give clear, practical advice.'},
        {'role':'user','content':f'Research results: {research}\n\nBased on this research, provide a recommendation.'}
    ],
    temperature=0.3)
print(step3.choices[0].message.content)

print('\nObserve: How does the agent decompose a task and execute it step by step?')

### Discussion
- How many sub-steps did the agent break the large task into? What's good about that?
- What's the biggest difference between an agent and a regular chatbot?
- In production, agents can call external tools (search engines, calculators, APIs) — can you imagine a scenario?

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| RAG System | Build a retrieval-augmented generation Q&A system from scratch |
| Keyword Search | Simple keyword-matching retrieval |
| Semantic Search | Use Embeddings for intelligent semantic matching |
| Agent | Understand how agents decompose tasks and execute step by step |

### Homework
1. Modify the knowledge base content — add your own documents and test the RAG system
2. Visit perplexity.ai to experience a production-grade RAG search engine
3. Try Google NotebookLM — upload a PDF and experience a personal knowledge base RAG